# Bài 56: Ước lượng độ sâu đơn mắt (Monocular Depth Estimation)

So khớp ảnh nổi ở Bài 22 khôi phục độ sâu từ *hai* ảnh bằng cách đo độ chênh lệch — một tín hiệu hình học trực tiếp. **Ước lượng độ sâu đơn mắt (monocular depth estimation)** đòi hỏi cùng bản đồ độ sâu theo từng điểm ảnh từ một ảnh *duy nhất*, hoàn toàn không có độ chênh lệch. Đây về bản chất là bài toán **đặt không chỉnh (ill-posed)**: vô số cảnh 3D tạo ra chính xác cùng một ảnh 2D (một chiếc ô tô đồ chơi chụp cận cảnh không thể phân biệt, đến từng điểm ảnh, với một chiếc ô tô thật chụp từ xa). Các mạng độ sâu đơn mắt hiện đại vẫn hoạt động được, bằng cách học các tiên nghiệm thống kê về thế giới &mdash; kích thước điển hình của đối tượng, phối cảnh, che khuất &mdash; từ các tập huấn luyện khổng lồ. Bài này xây dựng một mạng độ sâu nhỏ từ một tiên nghiệm như vậy (tín hiệu kích thước/phối cảnh), rồi cố tình phá vỡ tiên nghiệm đó để cho thấy chính xác tính đặt không chỉnh mà đoạn mở đầu đề cập thực sự gây hại ở đâu.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Một cảnh với tín hiệu độ sâu dựa trên kích thước

Các hình tròn ở các độ sâu khác nhau, với **độ sáng đồng đều có chủ đích** — thông tin duy nhất có được về độ sâu là *kích thước* của mỗi hình tròn (vật gần hơn được vẽ lớn hơn) và vị trí theo chiều dọc của nó (vật gần hơn nằm thấp hơn trong khung hình, một tín hiệu phối cảnh mặt đất đơn giản). Đây là thứ thay thế tổng hợp cho tiên nghiệm quan trọng nhất duy nhất mà một mạng độ sâu đơn mắt thực tế học được: một đối tượng thuộc một loại nhất định "nên" trông lớn đến đâu ở một khoảng cách nhất định.

In [ ]:
SIZE = 32

def make_scene(rng, size=SIZE, n_objects=4):
    img = np.zeros((size, size), dtype=np.float32)
    depth = np.full((size, size), 1.0, dtype=np.float32)  # background = far (depth = 1)
    depths_used = rng.uniform(0.2, 0.9, n_objects)
    depths_used.sort()
    for d in depths_used[::-1]:  # draw closest (smallest depth) last, so it occludes farther ones
        r = max(2, int(6 * (1 - d) + 1))       # closer (small d) -> larger radius
        cy = int(size * (0.3 + 0.6 * d))
        cx = rng.integers(r, size - r)
        yy, xx = np.mgrid[0:size, 0:size]
        m = ((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2
        img[m] = 0.8  # uniform intensity: size/position are the ONLY depth cues
        depth[m] = d
    img = np.clip(img + rng.normal(0, 0.03, img.shape), 0, 1).astype(np.float32)
    return img, depth

rng = np.random.default_rng(3)
N = 300
imgs, depths = [], []
for _ in range(N):
    im, d = make_scene(rng)
    imgs.append(im); depths.append(d)
imgs = np.array(imgs, dtype=np.float32)
depths = np.array(depths, dtype=np.float32)

split = int(0.85 * N)
Xtr, Dtr = imgs[:split], depths[:split]
Xte, Dte = imgs[split:], depths[split:]

fig, axes = plt.subplots(2, 4, figsize=(9, 4.5))
for i in range(4):
    axes[0, i].imshow(Xtr[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(Dtr[i], cmap='viridis_r', vmin=0.2, vmax=1.0); axes[1, i].axis('off')
axes[0, 0].set_title('image', fontsize=9, loc='left')
axes[1, 0].set_title('true depth (bright=near)', fontsize=9, loc='left')
plt.show()

## Một U-Net hồi quy độ sâu

Ước lượng độ sâu là hồi quy theo từng điểm ảnh chứ không phải phân loại theo từng điểm ảnh (Bài 42) — chính xác cùng kiến trúc mã-hóa-giải-mã-có-kết-nối-tắt, với một đầu ra liên tục duy nhất cho mỗi điểm ảnh thay vì một phân phối lớp.

In [ ]:
class UNetTiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.dec1 = nn.Sequential(nn.Conv2d(32 + 16, 16, 3, padding=1), nn.ReLU())
        self.out = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        f1 = self.enc1(x)
        f2 = self.enc2(self.pool(f1))
        d1 = self.dec1(torch.cat([self.up(f2), f1], dim=1))
        return torch.sigmoid(self.out(d1)).squeeze(1)  # depth in (0, 1)

torch.manual_seed(0)
model = UNetTiny()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
Xt = torch.tensor(Xtr).unsqueeze(1); Dt = torch.tensor(Dtr)
for _ in range(300):
    opt.zero_grad()
    loss = F.mse_loss(model(Xt), Dt)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred_te = model(torch.tensor(Xte).unsqueeze(1))

mae = (pred_te - torch.tensor(Dte)).abs().mean().item()
mean_depth_baseline = np.abs(Dte - Dtr.mean()).mean()
print(f'mean absolute depth error:                        {mae:.4f}  (depth range is [0.2, 1.0])')
print(f'baseline (predict the mean training depth everywhere): {mean_depth_baseline:.4f}')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(9, 6.5))
for i in range(4):
    axes[0, i].imshow(Xte[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(Dte[i], cmap='viridis_r', vmin=0.2, vmax=1.0); axes[1, i].axis('off')
    axes[2, i].imshow(pred_te[i], cmap='viridis_r', vmin=0.2, vmax=1.0); axes[2, i].axis('off')
for r, name in enumerate(['input', 'true depth', 'predicted depth']):
    axes[r, 0].set_title(name, fontsize=9, loc='left')
plt.tight_layout()
plt.show()

## Phá vỡ tiên nghiệm: bài toán "ô tô đồ chơi"

Mạng rõ ràng đã học được "hình tròn lớn hơn = gần hơn" — nhưng đó là một *mối liên hệ thống kê học được*, không phải một phép đo. Nếu kích thước của một đối tượng không tuân theo quy luật của phân phối huấn luyện (một mô hình thu nhỏ, một ảnh được chiếu, một vật thể thực sự khổng lồ), mạng không có cách nào biết được, vì ngay từ đầu nó chưa bao giờ có quyền truy cập vào hình học 3D thực. Hãy kiểm tra điều này trực tiếp: hai hình tròn có độ sâu thực *giống hệt nhau*, nhưng một hình có kích thước đúng với độ sâu đó và một hình có kích thước như thể nó ở gần hơn nhiều.

In [ ]:
def make_single_object(cx, cy, r, size=SIZE):
    img = np.zeros((size, size), dtype=np.float32)
    yy, xx = np.mgrid[0:size, 0:size]
    mask = ((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2
    img[mask] = 0.8
    return img, mask

true_depth = 0.8  # actually far away, in both cases below
normal_r = max(2, int(6 * (1 - true_depth) + 1))   # the radius a far object should have
faked_r = max(2, int(6 * (1 - 0.2) + 1))            # the radius as if it were very close instead

img_normal, mask_normal = make_single_object(16, 16, normal_r)
img_faked, mask_faked = make_single_object(16, 16, faked_r)

with torch.no_grad():
    pred_normal = model(torch.tensor(img_normal[None, None]))[0].numpy()
    pred_faked = model(torch.tensor(img_faked[None, None]))[0].numpy()

print(f'true depth in both cases: {true_depth}  (identical — nothing about the actual geometry changed)')
print(f'predicted depth, correctly-sized-for-its-depth object (r={normal_r}): {pred_normal[mask_normal].mean():.3f}')
print(f'predicted depth, oversized object (r={faked_r}, same true depth):     {pred_faked[mask_faked].mean():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img_normal, cmap='gray'); axes[0].set_title(f'normal size\npred depth={pred_normal[mask_normal].mean():.2f}', fontsize=9)
axes[1].imshow(img_faked, cmap='gray'); axes[1].set_title(f'oversized\npred depth={pred_faked[mask_faked].mean():.2f}', fontsize=9)
for ax in axes: ax.axis('off')
plt.show()

Mạng tự tin báo cáo hai độ sâu rất khác nhau cho hai đối tượng ở chính xác cùng một khoảng cách thực — nó không đo độ sâu, nó đang so khớp mẫu với "những thứ thuộc loại này thường trông lớn đến đâu ở khoảng cách này", và phép so khớp mẫu đó có thể bị đánh lừa bởi bất cứ thứ gì vi phạm các giả định của phân phối huấn luyện (mô hình thu nhỏ, ảnh chụp phối cảnh cưỡng bức (forced perspective), một vật thể có kích thước bất thường). Đây là kiểu thất bại thực tế, đã được ghi nhận rõ của các mạng độ sâu đơn mắt trong thực tế, và nó chính xác là điều "đặt không chỉnh nếu không có thông tin bổ sung" đã dự đoán ở đầu bài: riêng ảnh 2D không bao giờ chứa độ sâu thực, chỉ chứa các tín hiệu tương quan với nó trong dữ liệu huấn luyện điển hình.

Cách khắc phục thực tế mà các hệ thống thật dùng chính là thứ bài này đã mở đầu bằng cách đối chiếu: bất cứ khi nào có một phép đo hình học thứ hai — một camera thứ hai (ảnh nổi ở Bài 22, Bài 58), LiDAR, hoặc chuyển động camera đã biết (cấu trúc từ chuyển động, Bài 29) — nó nên được tin tưởng hơn một tiên nghiệm kích thước đơn mắt. Độ sâu đơn mắt có giá trị nhất chính ở những nơi không có các phép đo đó (một bức ảnh đơn lẻ, một khung hình video đơn lẻ không có đường cơ sở chuyển động), và kém đáng tin nhất chính ở nơi các giả định về phân phối huấn luyện của nó sụp đổ.

### Bài tập

1. Huấn luyện lại với độ sáng được khôi phục như một tín hiệu độ sâu (`img[m] = 0.4 + 0.5 * (1 - d)`, khớp với phiên bản đầu tiên của thí nghiệm này) thay vì cường độ đồng đều. Chạy lại phép thử đánh lừa — ảo giác dựa trên kích thước có yếu đi không, vì mô hình giờ có một tín hiệu thứ hai, không bị đánh lừa (độ sáng) để dựa vào?
2. Thước đo `silog` (scale-invariant log RMSE — RMSE log bất biến tỉ lệ) từ nghiên cứu độ sâu đơn mắt thực tế đo sai số sai khác một hệ số tỉ lệ nhân toàn cục, vì các dự đoán đơn mắt thường chỉ đúng *sai khác tỉ lệ*. Hãy cài đặt nó (`sqrt(mean((log(pred) - log(true))^2) - mean(log(pred) - log(true))^2)`) và so sánh với MAE đơn thuần — thứ hạng "dự đoán này tốt đến đâu" của mô hình có thay đổi giữa hai thước đo không?
3. Huấn luyện một mô hình thứ hai trên các cảnh với `n_objects` cố định bằng đúng 1 thay vì lấy theo mặc định của `make_scene`. Tiên nghiệm độ-sâu-từ-kích-thước của mô hình một đối tượng có chuyển giao sang các cảnh kiểm tra nhiều đối tượng ban đầu không, hay nó thất bại cụ thể khi các hình tròn che khuất nhau?